# L34 — Extreme-Condition and Degenerate Testing

**Module**: M10 | **Chapter**: 12 | **Lecture**: L34

## Learning Objectives
By the end of this notebook you will be able to:
1. Design extreme-condition tests that reveal model bugs at boundary inputs.
2. Verify conservation laws (mass balance, flow balance) as consistency checks.
3. Write parametric pytest fixtures that sweep boundary conditions automatically.
4. Distinguish a verified model from a validated one.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

Extreme conditions are the cheapest and most powerful verification tool: push inputs to
their limits, where the correct answer is known analytically, and confirm the model agrees.
---

In [ ]:
import math
import numpy as np
import simpy
import pytest

## 1. The Reference Model

We start with a correct M/M/1 implementation to establish the expected behaviors.

In [ ]:
def mm1_sim(lam, mu, n_customers=5000, warmup=500, seed=0):
    """Correct M/M/1 simulation. Returns dict of performance measures."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    waits, sojourns, service_times = [], [], []

    def customer():
        t_arrive = env.now
        with server.request() as req:
            yield req
            waits.append(env.now - t_arrive)
            svc = rng.exponential(1.0 / mu)
            service_times.append(svc)
            yield env.timeout(svc)
            sojourns.append(env.now - t_arrive)

    def arrivals():
        for _ in range(n_customers):
            env.process(customer())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()

    w = np.array(waits[warmup:])
    svc = np.array(service_times[warmup:])
    T = env.now

    return {
        'Wq':  w.mean() if len(w) else 0.0,
        'W':   np.array(sojourns[warmup:]).mean() if len(sojourns) > warmup else 0.0,
        'rho': svc.sum() / T if T > 0 else 0.0,
        'n_served': len(w),
        'T': T,
    }


def mm1_theory(lam, mu):
    rho = lam / mu
    Wq = rho / (mu - lam)
    W  = 1.0 / (mu - lam)
    return {'Wq': Wq, 'W': W, 'rho': rho}


# Quick sanity check at ρ=0.7
sim  = mm1_sim(0.7, 1.0, n_customers=10_000)
theo = mm1_theory(0.7, 1.0)
print(f"Sim  Wq={sim['Wq']:.3f}  W={sim['W']:.3f}  rho={sim['rho']:.3f}")
print(f"Theory Wq={theo['Wq']:.3f}  W={theo['W']:.3f}  rho={theo['rho']:.3f}")

## 2. Extreme Condition: Zero Arrivals (λ → 0)

When no customers arrive:
- `n_served = 0`
- `Wq = 0`, `W = 0` (or undefined — we return 0 by convention)
- Server utilisation `ρ = 0`
- Event calendar: only one arrival event scheduled, arrives at t→∞

In [ ]:
def test_zero_arrivals():
    result = mm1_sim(lam=0.0001, mu=1.0, n_customers=0, warmup=0)
    assert result['n_served'] == 0
    assert result['Wq'] == 0.0
    assert result['rho'] < 0.001
    print("PASS: zero arrivals")

test_zero_arrivals()

## 3. Extreme Condition: Light Load (ρ → 0)

At very low load, almost every customer finds the server idle. Therefore:
- `Wq ≈ 0`
- `W ≈ 1/μ` (sojourn ≈ service time only)
- `ρ ≈ λ/μ`

In [ ]:
for rho in [0.01, 0.05, 0.1]:
    lam = rho * 1.0  # mu=1
    sim  = mm1_sim(lam, 1.0, n_customers=5000, warmup=100)
    theo = mm1_theory(lam, 1.0)
    print(f"rho={rho:.2f}: sim Wq={sim['Wq']:.4f} (theory={theo['Wq']:.4f})  "
          f"W={sim['W']:.4f} (theory={theo['W']:.4f})")

print("\nAt low load: Wq→0, W→1/μ=1.0 — verified.")

## 4. Extreme Condition: Near-Critical Load (ρ → 1)

At ρ=0.99, Wq should be very large:
`Wq* = ρ/(μ−λ) = 0.99/(1.0−0.99) = 99.0` min

This requires a long simulation to see steady-state.

In [ ]:
rho = 0.95
sim_heavy = mm1_sim(rho, 1.0, n_customers=50_000, warmup=5000)
theo_heavy = mm1_theory(rho, 1.0)
print(f"rho={rho}: sim Wq={sim_heavy['Wq']:.2f} (theory={theo_heavy['Wq']:.2f})")
print(f"Relative error: {abs(sim_heavy['Wq'] - theo_heavy['Wq'])/theo_heavy['Wq']:.3f}")
print("\nNote: at high load, you need many more customers to estimate Wq accurately.")

## 5. Extreme Condition: Overloaded System (ρ > 1)

When λ > μ, the queue grows without bound. We should observe:
- Mean Wq growing linearly with simulation time
- Utilisation saturating at ~1.0
- No steady state exists

In [ ]:
import matplotlib.pyplot as plt

# Track Wq over time for rho=1.2 (unstable)
rng = np.random.default_rng(42)
env = simpy.Environment()
server = simpy.Resource(env, capacity=1)
LAM, MU = 1.2, 1.0
wait_times, arrival_times_log = [], []

def customer_overload():
    t0 = env.now
    arrival_times_log.append(t0)
    with server.request() as req:
        yield req
        wait_times.append(env.now - t0)
        yield env.timeout(rng.exponential(1.0/MU))

def arrivals_overload():
    for _ in range(2000):
        env.process(customer_overload())
        yield env.timeout(rng.exponential(1.0/LAM))

env.process(arrivals_overload())
env.run()

# Plot cumulative mean Wq over customers
cum_mean = np.cumsum(wait_times) / np.arange(1, len(wait_times)+1)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(cum_mean, color='tomato', lw=1)
ax.set_xlabel('Customer number')
ax.set_ylabel('Cumulative mean Wq (min)')
ax.set_title(f'Overloaded queue (λ={LAM}, μ={MU}, ρ={LAM/MU:.1f}) — no steady state')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Wq grows linearly: the queue never empties → no steady state.")

## 6. Conservation Law: Flow Balance

In a stable single-server queue over observation period T:
- Effective throughput: λ_eff = n_served / T
- Server utilisation: ρ = (total service time) / T
- These must satisfy: ρ = λ_eff / μ

Violations indicate bugs (double-counting, lost customers, clock errors).

In [ ]:
def test_flow_balance(lam=0.7, mu=1.0):
    sim = mm1_sim(lam, mu, n_customers=10_000, warmup=500)
    lam_eff = sim['n_served'] / sim['T']
    rho_from_lam = lam_eff / mu
    rho_direct   = sim['rho']
    diff = abs(rho_from_lam - rho_direct)
    print(f"ρ from throughput: {rho_from_lam:.4f}")
    print(f"ρ from busy time:  {rho_direct:.4f}")
    print(f"Difference: {diff:.6f}  {'PASS' if diff < 0.01 else 'FAIL'}")
    assert diff < 0.02, f"Flow balance violated: {diff:.4f}"

test_flow_balance()

## 7. Single-Customer Test: Exact Trace Verification

With exactly 1 customer:
- Customer arrives at t = interarrival_time
- Finds server idle → Wq = 0
- W = service_time
- ρ = service_time / T

In [ ]:
rng_test = np.random.default_rng(7)
ia_time  = rng_test.exponential(1.0 / 0.8)   # interarrival
svc_time = rng_test.exponential(1.0 / 1.0)   # service
print(f"Pre-computed: ia={ia_time:.4f}, svc={svc_time:.4f}")
print(f"Expected Wq=0.0, W={svc_time:.4f}")

sim1 = mm1_sim(0.8, 1.0, n_customers=1, warmup=0, seed=7)
print(f"Simulated:    Wq={sim1['Wq']:.4f}, W={sim1['W']:.4f}")
print(f"Wq==0: {math.isclose(sim1['Wq'], 0.0, abs_tol=1e-9)}")

## 8. Parametric Boundary Test Suite

A well-designed test suite sweeps multiple boundary conditions automatically.

In [ ]:
import pandas as pd

# Sweep ρ ∈ {0.1, 0.3, 0.5, 0.7, 0.9} and check relative error
mu = 1.0
rho_values = [0.1, 0.3, 0.5, 0.7, 0.9]
rows = []
for rho in rho_values:
    lam = rho * mu
    sim  = mm1_sim(lam, mu, n_customers=20_000, warmup=1000)
    theo = mm1_theory(lam, mu)
    rel_err = abs(sim['Wq'] - theo['Wq']) / max(theo['Wq'], 1e-9)
    rows.append({
        'rho': rho,
        'Wq_sim':   round(sim['Wq'], 4),
        'Wq_theory': round(theo['Wq'], 4),
        'rel_error': round(rel_err, 4),
        'PASS': rel_err < 0.10,
    })

print(pd.DataFrame(rows).to_string(index=False))
print("\nAll pass at <10% relative error? ",
      all(r['PASS'] for r in rows))

---
## Try It Yourself

1. **Inventory extreme conditions**: Write three boundary tests for the `SSInventory` model:
   (a) demand_rate=0 (stock never depletes — no orders placed, on-hand stays at S),
   (b) s=S (order immediately whenever stock drops below S — maximum order frequency),
   (c) lead_time=0 (replenishment is instantaneous — no stockouts possible).
   What is the expected behaviour in each case?

2. **M/M/2 boundary**: Implement an M/M/2 queue and test:
   (a) When only one customer is ever in the system, both servers should never be busy simultaneously.
   (b) At ρ=0, both servers are always idle.
   (c) The combined throughput cannot exceed 2μ.

3. **Reneging boundary**: In a queue with patience=∞, no customers should renege. In a queue with patience=0, all customers should renege immediately unless the server is idle. Write assertions for both cases.